# Stage 2: Baseline Tokenization

This notebook establishes baseline tokenization behavior on PubMedQA.

Tokenizers compared:
- `bert-base-uncased` (WordPiece, general domain)
- `dmis-lab/biobert-base-cased-v1.1` (biomedical domain)
- `gpt2` (BPE)

Computed metrics:
- average number of tokens per sample,
- fertility (tokens per word),
- UNK rate,
- average token length in characters.

In [1]:
import json
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import BertTokenizer, GPT2Tokenizer

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
RANDOM_SEED = 42
SAMPLE_SIZE = 300
RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

TOKENIZER_SPECS = {
    "bert_base": "bert-base-uncased",
    "biobert": "dmis-lab/biobert-base-cased-v1.1",
    "gpt2": "gpt2",
}

TARGET_TERMS = [
    "nephrolithiasis",
    "gastroesophageal",
    "dermographism",
    "electrocardiogram",
    "immunohistochemistry",
    "acetylcholinesterase",
    "hypercholesterolemia",
    "hepatocellular",
    "neurodegenerative",
    "myocardial",
]

In [3]:
def get_context_text(context_field):
    if isinstance(context_field, dict):
        contexts = context_field.get("contexts", [])
        if isinstance(contexts, list):
            return " ".join(str(part) for part in contexts)
    if isinstance(context_field, list):
        return " ".join(str(part) for part in context_field)
    return str(context_field)


def count_words(text):
    return len(text.split())


def get_unk_token_candidates(tokenizer):
    candidates = set()
    if tokenizer.unk_token is not None:
        candidates.add(tokenizer.unk_token)
    if tokenizer.unk_token_id is not None:
        unk_from_id = tokenizer.convert_ids_to_tokens([tokenizer.unk_token_id])[0]
        candidates.add(unk_from_id)
    return candidates


def clean_token(token):
    return re.sub(r"^[\W_]+|[\W_]+$", "", token)


def evaluate_tokenizer(tokenizer, texts):
    unk_candidates = get_unk_token_candidates(tokenizer)
    token_counts = []
    word_counts = []
    unk_count = 0
    total_tokens = 0
    char_lengths = []
    vocab_counter = Counter()

    for text in texts:
        words = count_words(text)
        encoding = tokenizer(text, add_special_tokens=True)
        ids = encoding["input_ids"]
        tokens = tokenizer.convert_ids_to_tokens(ids)
        token_counts.append(len(tokens))
        word_counts.append(words)

        for token in tokens:
            total_tokens += 1
            vocab_counter[token] += 1
            if token in unk_candidates:
                unk_count += 1
            if token not in tokenizer.all_special_tokens:
                normalized = clean_token(token)
                if normalized:
                    char_lengths.append(len(normalized))

    avg_tokens = float(np.mean(token_counts))
    fertility = float(np.sum(token_counts) / max(np.sum(word_counts), 1))
    unk_rate = float(unk_count / max(total_tokens, 1))
    avg_token_length = float(np.mean(char_lengths)) if char_lengths else 0.0

    return {
        "avg_tokens_per_sample": avg_tokens,
        "fertility": fertility,
        "unk_rate": unk_rate,
        "avg_token_length_chars": avg_token_length,
        "top_tokens": vocab_counter.most_common(30),
    }


def token_split_examples(tokenizer, terms):
    rows = []
    for term in terms:
        ids = tokenizer(term, add_special_tokens=False)["input_ids"]
        pieces = tokenizer.convert_ids_to_tokens(ids)
        rows.append({"term": term, "pieces": pieces, "n_pieces": len(pieces)})
    return pd.DataFrame(rows)


In [4]:
dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled")["train"]

df = pd.DataFrame({
    "question": dataset["question"],
    "context_text": [get_context_text(x) for x in dataset["context"]],
    "label": dataset["final_decision"],
})

df["text"] = df["question"].fillna("") + " [SEP] " + df["context_text"].fillna("")
sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
sample_texts = sample_df["text"].tolist()

print(f"Total labeled samples: {len(df)}")
print(f"Evaluation sample size: {len(sample_df)}")

Total labeled samples: 1000
Evaluation sample size: 300


In [5]:
tokenizers = {
    "bert_base": BertTokenizer.from_pretrained(TOKENIZER_SPECS["bert_base"]),
    "biobert": BertTokenizer.from_pretrained(TOKENIZER_SPECS["biobert"]),
    "gpt2": GPT2Tokenizer.from_pretrained(TOKENIZER_SPECS["gpt2"]),
}
list(tokenizers.keys())

['bert_base', 'biobert', 'gpt2']

In [6]:
metrics_rows = []
token_split_tables = []
top_tokens_json = {}

for name, tokenizer in tokenizers.items():
    metrics = evaluate_tokenizer(tokenizer, sample_texts)
    metrics_rows.append({
        "tokenizer_name": name,
        "model_id": TOKENIZER_SPECS[name],
        "avg_tokens_per_sample": metrics["avg_tokens_per_sample"],
        "fertility": metrics["fertility"],
        "unk_rate": metrics["unk_rate"],
        "avg_token_length_chars": metrics["avg_token_length_chars"],
        "stage": "stage2_baseline_tokenization",
    })
    top_tokens_json[name] = metrics["top_tokens"]

    term_table = token_split_examples(tokenizer, TARGET_TERMS)
    term_table["tokenizer_name"] = name
    token_split_tables.append(term_table)

metrics_df = pd.DataFrame(metrics_rows).sort_values(by="fertility")
examples_df = pd.concat(token_split_tables, ignore_index=True)

metrics_df

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (613 > 512). Running this sequence through the model will result in indexing errors


,tokenizer_name,model_id,avg_tokens_per_sample,fertility,unk_rate,avg_token_length_chars,stage
2,gpt2,gpt2,313.383333,1.492136,0.0,5.010264,stage2_baseline_tokenization
0,bert_base,bert-base-uncased,323.886667,1.542146,0.0,4.287131,stage2_baseline_tokenization
1,biobert,dmis-lab/biobert-base-cased-v1.1,339.490000,1.616439,0.0,4.052037,stage2_baseline_tokenization


In [7]:
metrics_path = os.path.join(RESULTS_DIR, "metrics.csv")
examples_path = os.path.join(RESULTS_DIR, "examples.md")
splits_path = os.path.join(RESULTS_DIR, "stage2_term_token_splits.csv")
top_tokens_path = os.path.join(RESULTS_DIR, "stage2_top_tokens.json")

existing_metrics = pd.read_csv(metrics_path) if os.path.exists(metrics_path) and os.path.getsize(metrics_path) > 0 else pd.DataFrame()
if not existing_metrics.empty:
    existing_metrics = existing_metrics[existing_metrics["stage"] != "stage2_baseline_tokenization"]

final_metrics = pd.concat([existing_metrics, metrics_df], ignore_index=True)
final_metrics.to_csv(metrics_path, index=False)
examples_df.to_csv(splits_path, index=False)

with open(top_tokens_path, "w", encoding="utf-8") as f:
    json.dump(top_tokens_json, f, indent=2)

with open(examples_path, "w", encoding="utf-8") as f:
    f.write("# Tokenization Examples\n\n")
    f.write("## Stage 2: Baseline Tokenization\n\n")
    for tokenizer_name in metrics_df["tokenizer_name"].tolist():
        f.write(f"### {tokenizer_name}\n\n")
        f.write("| term | pieces | n_pieces |\n")
        f.write("|---|---|---:|\n")
        section = examples_df[examples_df["tokenizer_name"] == tokenizer_name][["term", "pieces", "n_pieces"]]
        for _, row in section.iterrows():
            pieces = str(row["pieces"]).replace("|", "\\|")
            term = str(row["term"]).replace("|", "\\|")
            f.write(f"| {term} | {pieces} | {int(row['n_pieces'])} |\n")
        f.write("\n")

print(f"Saved metrics to: {metrics_path}")
print(f"Saved examples markdown to: {examples_path}")
print(f"Saved token splits table to: {splits_path}")
print(f"Saved top tokens json to: {top_tokens_path}")

Saved metrics to: ../results/metrics.csv
Saved examples markdown to: ../results/examples.md
Saved token splits table to: ../results/stage2_term_token_splits.csv
Saved top tokens json to: ../results/stage2_top_tokens.json


## Interpretation Checklist

When reviewing Stage 2 outputs, compare:
- fertility differences across tokenizers,
- whether biomedical terms are split into many pieces,
- UNK behavior,
- average token length and top frequent token patterns.